# BIRD-SQL Evaluation — EX & VES cho các model Ollama

Chạy `evaluation.py` (Execution Accuracy) và `evaluation_ves.py` (Valid Efficiency Score)
cho 3 model đã có kết quả: `deepseek-coder-v2`, `gemma3`, `qwen2.5-coder`.

Gọi qua `!python` (subprocess thật) thay vì `import` trực tiếp — 2 script này dùng
`multiprocessing.Pool`, import thẳng vào kernel Jupyter trên Windows sẽ lỗi spawn.

In [2]:
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
LLM_DIR = PROJECT_ROOT / "src" / "DAMO-ConvAI" / "bird" / "llm"
BIRD_DATA_DIR = PROJECT_ROOT / "src" / "DAMO-ConvAI" / "bird" / "data"

MODELS = ["ollama_deepseekcoder_output", "ollama_gemma3_output", "ollama_gwen25_output"]

for m in MODELS:
    p = LLM_DIR / "exp_result" / m / "predict_dev.json"
    print(m, "->", "OK" if p.exists() else "MISSING", p)

ollama_deepseekcoder_output -> OK d:\AI Learning\Paper\LLM\src\DAMO-ConvAI\bird\llm\exp_result\ollama_deepseekcoder_output\predict_dev.json
ollama_gemma3_output -> OK d:\AI Learning\Paper\LLM\src\DAMO-ConvAI\bird\llm\exp_result\ollama_gemma3_output\predict_dev.json
ollama_gwen25_output -> OK d:\AI Learning\Paper\LLM\src\DAMO-ConvAI\bird\llm\exp_result\ollama_gwen25_output\predict_dev.json


In [3]:
def run_eval_script(script_name: str, predicted_dir_name: str, num_cpus: int = 4, meta_time_out: float = 30.0,
                     extra_args=None) -> str:
    """Run evaluation.py or evaluation_ves.py as a subprocess (avoids Windows multiprocessing-in-kernel issues)."""
    cmd = [
        sys.executable, f"src/{script_name}",
        "--db_root_path", str(BIRD_DATA_DIR / "dev_databases") + os.sep,
        "--predicted_sql_path", str(LLM_DIR / "exp_result" / predicted_dir_name) + os.sep,
        "--ground_truth_path", str(LLM_DIR / "data") + os.sep,
        "--data_mode", "dev",
        "--num_cpus", str(num_cpus),
        "--meta_time_out", str(meta_time_out),
        "--mode_gt", "gt",
        "--mode_predict", "gpt",
        "--diff_json_path", str(BIRD_DATA_DIR / "dev.json"),
    ]
    if extra_args:
        cmd += extra_args
    result = subprocess.run(cmd, cwd=str(LLM_DIR), capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{predicted_dir_name} / {script_name}] STDERR:\n", result.stderr[-2000:])
    return result.stdout


SCORE_RE = re.compile(r"(accuracy|ves)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)")


def parse_score(stdout: str):
    match = SCORE_RE.search(stdout)
    if not match:
        return None
    _, simple, moderate, challenging, total = match.groups()
    return [float(simple), float(moderate), float(challenging), float(total)]

In [ ]:
ex_results = {}
ves_results = {}

for m in MODELS:
    print(f"\n===== {m} : EX =====")
    ex_out = run_eval_script("evaluation.py", m)
    print(ex_out[-600:])
    ex_results[m] = parse_score(ex_out)

    print(f"\n===== {m} : VES (iterate_num=5) =====")
    ves_out = run_eval_script("evaluation_ves.py", m, extra_args=["--iterate_num", "5"])
    print(ves_out[-600:])
    ves_results[m] = parse_score(ves_out)


===== ollama_deepseekcoder_output : EX =====
start calculate
                     simple               moderate             challenging          total               
count                925                  464                  145                  1534                
======================================    ACCURACY    =====================================
accuracy             50.81                31.25                22.76                42.24               
Finished evaluation


===== ollama_deepseekcoder_output : VES (iterate_num=5) =====
start calculate
                     simple               moderate             challenging          total               
count                925                  464                  145                  1534                
=========================================    VES   ========================================
ves                  53.63                31.30                22.43                43.93               
Finished evaluation


==

In [ ]:
cols = ["simple", "moderate", "challenging", "total"]

ex_df = pd.DataFrame(ex_results, index=cols).T
ex_df.columns = [f"EX_{c}" for c in ex_df.columns]

ves_df = pd.DataFrame(ves_results, index=cols).T
ves_df.columns = [f"VES_{c}" for c in ves_df.columns]

summary = ex_df.join(ves_df)
summary

In [ ]:
# Lưu bảng kết quả ra CSV để chèn vào báo cáo
OUT_CSV = LLM_DIR / "exp_result" / "summary_ex_ves.csv"
summary.to_csv(OUT_CSV)
print("Saved:", OUT_CSV)